In [3]:
# configuracion: imports, constantes y carga de DS2OS

import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings('ignore')

RANDOM_SEED = 42  # GAP: no reportado en el paper

# 8 clases del dataset DS2OS
CLASS_MAP = {
    'normal'                        : 'Normal',
    'anomalous(DoSattack)'          : 'DoS',
    'anomalous(scan)'               : 'Scan',
    'anomalous(malitiousControl)'   : 'MaliciousControl',
    'anomalous(malitiousOperation)' : 'MaliciousOperation',
    'anomalous(spying)'             : 'Spying',
    'anomalous(dataProbing)'        : 'DataProbing',
    'anomalous(wrongSetUp)'         : 'WrongSetUp',
}
CLASS_TO_INT = {v: i for i, v in enumerate(CLASS_MAP.values())}
INT_TO_CLASS = {v: k for k, v in CLASS_TO_INT.items()}

# Busca DS2OS.csv en rutas habituales
SEARCH_PATHS = [
    Path("DS2OS.csv"),
    Path.home() / "Downloads" / "DS2OS.csv",
    (Path.home() / ".cache" / "kagglehub" / "datasets" / "libamariyam" / "ds2os-dataset" / "versions" / "1" / "DS2OS.csv"),
]

CSV_PATH = next((p for p in SEARCH_PATHS if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError(
        "DS2OS.csv no encontrado. Colócalo en el directorio actual o en ~/Downloads/"
    )

df_raw = pd.read_csv(CSV_PATH)
print(f"Dataset cargado: {CSV_PATH}")
print(f"Shape: {df_raw.shape}  — {df_raw.shape[1]} columnas, {df_raw.shape[0]:,} instancias")


Dataset cargado: C:\Users\user\.cache\kagglehub\datasets\libamariyam\ds2os-dataset\versions\1\DS2OS.csv
Shape: (357952, 13)  — 13 columnas, 357,952 instancias


In [4]:
# preprocesamiento y seleccion, pipeline de 6 pasos (seccion V del paper)

df = df_raw.copy()

# Paso 1 — NaN en accessedNodeType → "Malicious" (paper Sec. V-A)
df['accessedNodeType'] = df['accessedNodeType'].fillna('Malicious')

# Paso 2 — Limpiar columna value: texto → numérico
df['value'] = df['value'].replace({'False': 0, 'True': 1, 'Twenty': 20, 'none': 0})
df['value'] = pd.to_numeric(df['value'], errors='coerce').fillna(0)

# Paso 3 — Eliminar timestamp (relación nominal con la normalidad, paper Sec. V-B)
df = df.drop(columns=['timestamp'])

# Paso 4 — Codificar etiquetas de clase
df['y'] = df['normality'].map(CLASS_MAP).map(CLASS_TO_INT).astype(np.int8)
df = df.drop(columns=['normality'])

# Paso 5 — Label Encoding sobre columnas categóricas (paper: ahorra espacio frente a One-Hot)
# GAP: el paper menciona MCA, sustituido por Label Encoding directo
CAT_COLS = [
    'sourceID', 'sourceAddress', 'sourceType', 'sourceLocation',
    'destinationServiceAddress', 'destinationServiceType',
    'destinationLocation', 'accessedNodeAddress', 'accessedNodeType', 'operation',
]
for col in CAT_COLS:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

FEATURE_COLS = CAT_COLS + ['value']  # 11 features
X = df[FEATURE_COLS].values.astype(np.float32)
y = df['y'].values

# Paso 6 — Split 80/20 estratificado + StandardScaler post-split
# GAP: ratio no especificado en el paper; 80/20 es estándar para DS2OS
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y
)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Resumen del split
print(f"Features : {len(FEATURE_COLS)}")
print(f"Total    : {len(y):>8,} instancias")
print(f"Train    : {len(X_train):>8,} instancias (80%)")
print(f"Test     : {len(X_test):>8,} instancias (20%)")

# Distribución por clase: total, train y test
print(f"\n{'Clase':<22} {'Total':>8} {'%':>5}  {'Train':>8}  {'Test':>6}")
print(f"{'─'*22} {'─'*8} {'─'*5}  {'─'*8}  {'─'*6} ")
for cls_id, cls_name in INT_TO_CLASS.items():
    n_total = (y       == cls_id).sum()
    n_train = (y_train == cls_id).sum()
    n_test  = (y_test  == cls_id).sum()
    print(
        f"{cls_name:<22} {n_total:>8,} {100*n_total/len(y):>4.1f}%"
        f"  {n_train:>8,} {100*n_train/len(y_train):>4.1f}%"
        f"  {n_test:>6,} {100*n_test/len(y_test):>4.1f}%"
    )


Features : 11
Total    :  357,952 instancias
Train    :  286,361 instancias (80%)
Test     :   71,591 instancias (20%)

Clase                     Total     %     Train    Test
────────────────────── ──────── ─────  ────────  ────── 
Normal                  347,935 97.2%   278,347 97.2%  69,588 97.2%
DoS                       5,780  1.6%     4,624  1.6%   1,156  1.6%
Scan                      1,547  0.4%     1,237  0.4%     310  0.4%
MaliciousControl            889  0.2%       711  0.2%     178  0.2%
MaliciousOperation          805  0.2%       644  0.2%     161  0.2%
Spying                      532  0.1%       426  0.1%     106  0.1%
DataProbing                 342  0.1%       274  0.1%      68  0.1%
WrongSetUp                  122  0.0%        98  0.0%      24  0.0%


In [5]:
# definicion y entrenamiento: 7 clasificadores individuales + AdaBoost
# hiperparametros del paper: KNN euclidea, SVM rbf gamma=0.1, ANN sigmoide+sgd, resto defaults
# AdaBoost no viene especificado en el paper, se usa DT(max_depth=3) n=100

CLASSIFIERS = {
    'KNN'     : KNeighborsClassifier(metric='euclidean'),
    'LDA'     : LinearDiscriminantAnalysis(),
    'DT'      : DecisionTreeClassifier(random_state=RANDOM_SEED),
    'RF'      : RandomForestClassifier(random_state=RANDOM_SEED),
    'LR'      : LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    'SVM'     : SVC(kernel='rbf', gamma=0.1, random_state=RANDOM_SEED),
    'ANN'     : MLPClassifier(activation='logistic', solver='sgd', max_iter=500, random_state=RANDOM_SEED),
    'AdaBoost': AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED),
                    n_estimators=100, random_state=RANDOM_SEED, algorithm='SAMME'),
}

predictions = {}
for name, clf in CLASSIFIERS.items():
    print(f"Entrenando {name}...", end=' ', flush=True)
    clf.fit(X_train, y_train)
    predictions[name] = clf.predict(X_test)
    print("listo.")

print("\nEntrenamiento completado.")


Entrenando KNN... listo.
Entrenando LDA... listo.
Entrenando DT... listo.
Entrenando RF... listo.
Entrenando LR... listo.
Entrenando SVM... listo.
Entrenando ANN... listo.
Entrenando AdaBoost... listo.

Entrenamiento completado.


In [6]:
# resultados: Tabla I del paper + analisis por clase de AdaBoost

# Valores de referencia — Tabla I del paper (Khare & Totaro, 2020)
PAPER_RESULTS = {
    'KNN'     : {'precision': 0.98, 'recall': 0.98, 'f1': 0.98, 'accuracy': 0.98},
    'LDA'     : {'precision': 0.98, 'recall': 0.98, 'f1': 0.98, 'accuracy': 0.98},
    'DT'      : {'precision': 0.97, 'recall': 0.97, 'f1': 0.97, 'accuracy': 0.97},
    'RF'      : {'precision': 0.97, 'recall': 0.97, 'f1': 0.97, 'accuracy': 0.97},
    'LR'      : {'precision': 0.96, 'recall': 0.96, 'f1': 0.96, 'accuracy': 0.96},
    'SVM'     : {'precision': 0.97, 'recall': 0.97, 'f1': 0.97, 'accuracy': 0.97},
    'ANN'     : {'precision': 0.97, 'recall': 0.97, 'f1': 0.97, 'accuracy': 0.97},
    'AdaBoost': {'precision': 0.99, 'recall': 0.99, 'f1': 0.99, 'accuracy': 0.99},
}

# Tabla I: métricas globales (weighted average)
print("TABLA I — Khare & Totaro (2020): Clasificadores vs. Réplica")
print(f"{'─'*72}")
print(f"{'Modelo':<10} {'Prec':>6} {'Rec':>6} {'F1':>6} {'Acc':>6}  "
      f"{'Prec*':>6} {'F1*':>6} {'Δ F1':>7}")
print(f"{'─'*10} {'─'*6} {'─'*6} {'─'*6} {'─'*6}  "
      f"{'─'*6} {'─'*6} {'─'*7}")

for name, y_pred in predictions.items():
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score   (y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score       (y_test, y_pred, average='weighted', zero_division=0)
    acc  = accuracy_score (y_test, y_pred)

    ref  = PAPER_RESULTS.get(name, {})
    p_p  = ref.get('precision', float('nan'))
    p_f1 = ref.get('f1',        float('nan'))
    delta = f1 - p_f1

    print(f"{name:<10} {prec:.3f}  {rec:.3f}  {f1:.3f}  {acc:.3f}  "
          f"{p_p:.2f}   {p_f1:.2f}  {delta:+.3f}")

print(f"{'─'*72}")
print("* Valores del paper (Tabla I). Δ F1 = réplica − paper.")


TABLA I — Khare & Totaro (2020): Clasificadores vs. Réplica
────────────────────────────────────────────────────────────────────────
Modelo       Prec    Rec     F1    Acc   Prec*    F1*    Δ F1
────────── ────── ────── ────── ──────  ────── ────── ───────
KNN        0.995  0.995  0.994  0.995  0.98   0.98  +0.014
LDA        0.981  0.981  0.979  0.981  0.98   0.98  -0.001
DT         0.995  0.995  0.994  0.995  0.97   0.97  +0.024
RF         0.995  0.995  0.994  0.995  0.97   0.97  +0.024
LR         0.987  0.989  0.987  0.989  0.96   0.96  +0.027
SVM        0.995  0.995  0.994  0.995  0.97   0.97  +0.024
ANN        0.989  0.990  0.988  0.990  0.97   0.97  +0.018
AdaBoost   0.989  0.989  0.986  0.989  0.99   0.99  -0.004
────────────────────────────────────────────────────────────────────────
* Valores del paper (Tabla I). Δ F1 = réplica − paper.
